# Blind transfer of the field-adapted students to External datasets 1 and 2

The field-adapted student **S1** was re-distilled on probe-labelled photographs from four Habiganj estates
(Section IV-Q). External dataset 1 (East West University) and External dataset 2 (Allynugger, Moulvibazar)
played no part in that adaptation, so scoring S1 on them is **blind transfer** for the adapted model.

* Models: the nine saved students `experiment_F/student_{S0,S1,S2}_rep{0,1,2}.pt` (inference only, nothing is trained).
* Inputs: exactly as Table 15 — External dataset 1 from its YOLO leaf boxes, External dataset 2 on the whole frame
  (same crop code as `external_test.ipynb`, JPEG q95 round trip).
* Threshold: each student's own 95%-sensitivity threshold on **our** test split, from the scores saved in
  `experiment_F_scores.npz`, fixed before any external image is scored.
* Checks: the saved test-split scores reproduce the published AUCs; re-scoring 96 of our test images with the loaded
  S0/S1 weights matches the saved scores (stops otherwise).

Output: `results_revision_v1/experiment_F/external_blind_transfer.json` — upload it.
The last cell also lists `external_test/` so you can upload the final External dataset 1 files (item 2).

Runtime: ~5 min with a GPU, ~15 min on CPU (inference only).

In [ ]:
# CELL 0 — CONFIG (same paths as external_test.ipynb and experiment_F_robustness.ipynb)
DRIVE    = '/content/drive/MyDrive'
TEA_ROOT = f'{DRIVE}/Tea Leaf Dataset'
RES      = f'{DRIVE}/TeaLeaf_DINOv2_Study/results_revision_v1'
SPLIT    = f'{RES}/_split.json'
OUTF     = f'{RES}/experiment_F'                      # students + experiment_F_scores.npz
EXT_OUT  = f'{RES}/external_test'                     # final External dataset 1 files live here
EWU_DIR  = f'{DRIVE}/East West University dataset'
ALLY_DIR = f'{DRIVE}/Allynugger'
OUT      = f'{OUTF}/external_blind_transfer.json'
PAD, IMG, JPEG_Q, TARGET_SENS = 0.06, 224, 95, 0.95
NAMES, REPS = ('S0', 'S1', 'S2'), (0, 1, 2)


In [ ]:
# CELL 1 — setup
from google.colab import drive; drive.mount('/content/drive')
!pip -q install pyyaml
import os, io, re, glob, json, zipfile, shutil, time, warnings
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from collections import Counter
import numpy as np, cv2, yaml, torch, torch.nn as nn
import torchvision.models as tvm, torchvision.transforms as T
from PIL import Image
from scipy import ndimage
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


In [ ]:
# CELL 2 — (verbatim from external_test.ipynb) preprocessing (same recipe as the paper: Otsu on HSV V|S, largest component,
#           fill holes, bbox crop with 6% padding, resize 224x224, JPEG q95 round-trip)
def leaf_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0)
    lab, n = ndimage.label(m)
    if n == 0:
        return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def box_of(mask):
    ys, xs = np.where(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

def crop_box(rgb, box, pad=PAD, size=IMG):
    H, W = rgb.shape[:2]; x0, y0, x1, y1 = box
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0 = int(max(0, x0 - px)), int(max(0, y0 - py))
    x1, y1 = int(min(W, x1 + px)), int(min(H, y1 + py))
    im = Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q)       # match the cached-crop pipeline
    return Image.open(io.BytesIO(buf.getvalue())).convert('RGB')

def load_rgb(p):
    return np.array(Image.open(p).convert('RGB'))

def otsu_crop(rgb):
    m = leaf_mask(rgb)
    return crop_box(rgb, box_of(m)), float(m.mean())

def whole_crop(rgb):
    H, W = rgb.shape[:2]
    return crop_box(rgb, (0, 0, W, H), pad=0.0), 1.0


In [ ]:
# CELL 3 — (verbatim from external_test.ipynb) EWU: unzip Roboflow export, read YOLO boxes, drop Roboflow augmented copies
EWU_X = '/content/ewu'
if not os.path.exists(EWU_X):
    z = [p for p in glob.glob(f'{EWU_DIR}/*.zip')]
    assert z, 'no zip found in ' + EWU_DIR
    zipfile.ZipFile(z[0]).extractall(EWU_X)
yml = glob.glob(f'{EWU_X}/**/data.yaml', recursive=True) or glob.glob(f'{EWU_DIR}/data.yaml')
names = yaml.safe_load(open(yml[0]))['names']
names = list(names.values()) if isinstance(names, dict) else list(names)
print('EWU classes:', names)

def rf_stem(fname):          # Roboflow: <orig>_jpg.rf.<hash>.jpg  -> <orig>
    m = re.match(r'^(.*?)_(jpe?g|png|bmp|webp)\.rf\.', os.path.basename(fname), re.I)
    return m.group(1) if m else os.path.splitext(os.path.basename(fname))[0]

def split_rank(p):           # prefer un-augmented splits when de-duplicating
    return 0 if '/test/' in p else 1 if '/valid/' in p else 2

imgs = sorted([p for p in glob.glob(f'{EWU_X}/**/*', recursive=True) if p.lower().endswith(IMG_EXT)],
              key=lambda p: (split_rank(p), p))   # deterministic: same copy kept every run
seen, EWU = set(), []
for p in imgs:
    st = rf_stem(p)
    if st in seen: continue
    lbl = re.sub(r'/images/', '/labels/', p); lbl = os.path.splitext(lbl)[0] + '.txt'
    if not os.path.exists(lbl): continue
    seen.add(st)
    for line in open(lbl).read().strip().splitlines():
        v = line.split()
        if len(v) != 5: continue          # skip polygon rows
        EWU.append(dict(path=p, cls=names[int(v[0])], yolo=tuple(map(float, v[1:]))))
print('EWU: %d unique source images, %d leaf boxes (of %d image files incl. augmentations)'
      % (len(seen), len(EWU), len(imgs)))
from collections import Counter; print(Counter(r['cls'] for r in EWU))

def yolo_to_box(yolo, W, H):
    cx, cy, w, h = yolo
    return (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H


In [ ]:
# CELL 4 — (verbatim from external_test.ipynb) Allynugger: one folder per class, copied to local disk first (Drive reads are slow)
ALLY_LOCAL = '/content/ally_local'
src_files = [p for p in sorted(glob.glob(f'{ALLY_DIR}/*/*')) if p.lower().endswith(IMG_EXT)]
def _cp(p):
    d = os.path.join(ALLY_LOCAL, *p.split('/')[-2:])
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(p, d)
    return d
t0 = time.time()
with ThreadPoolExecutor(16) as ex: local_files = list(ex.map(_cp, src_files))
ALLY = [dict(path=p, cls=os.path.basename(os.path.dirname(p))) for p in local_files]
print('Allynugger: %s  (copied in %.0fs)' % (Counter(r['cls'] for r in ALLY), time.time() - t0))


In [ ]:
# CELL 5 — crops exactly as Table 15 (External dataset 1: leaf boxes; External dataset 2: whole frame)
is_healthy = lambda c: 'healthy' in c.lower()
def _one(args):
    r, mode = args
    try:
        rgb = load_rgb(r['path']); H, W = rgb.shape[:2]
        if mode == 'box':
            b = yolo_to_box(r['yolo'], W, H); im = crop_box(rgb, b)
        else:
            im, _ = whole_crop(rgb)
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q); return buf.getvalue()
    except Exception as e:
        return None
def build(records, mode):
    try:
        with ProcessPoolExecutor(os.cpu_count()) as ex: res = list(ex.map(_one, [(r, mode) for r in records], chunksize=16))
    except Exception:
        with ThreadPoolExecutor(8) as ex: res = list(ex.map(_one, [(r, mode) for r in records]))
    keep = [i for i, b in enumerate(res) if b is not None]
    crops = [Image.open(io.BytesIO(res[i])).convert('RGB') for i in keep]
    recs = [records[i] for i in keep]
    return dict(crops=crops, y=np.array([0 if is_healthy(r['cls']) else 1 for r in recs]),
                cls=np.array([r['cls'] for r in recs]))
SETS = {'External_1': build(EWU, 'box'), 'External_2': build(ALLY, 'whole')}
for k, S in SETS.items():
    print('%-10s n=%5d  healthy=%4d  classes %s' % (k, len(S['y']), (S['y'] == 0).sum(), dict(Counter(S['cls']))))
print('Table 15 reports External dataset 1: 3,007 (905 healthy); External dataset 2: 883 (524 healthy).')


In [ ]:
# CELL 6 — the nine saved students, and the thresholds fixed on OUR test split from the saved scores
def load_student(path):
    m = tvm.mobilenet_v3_small(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(device).eval()
SC = np.load(f'{OUTF}/experiment_F_scores.npz'); dte = SC['tealeaf_test_d']
def thr_at_sens(s, d, t=TARGET_SENS): return float(np.sort(s[d == 1])[int(np.floor((1 - t) * (d == 1).sum()))])
STU, THR = {}, {}
RJ = json.load(open(f'{OUTF}/experiment_F_robustness.json'))
for n in NAMES:
    for r in REPS:
        p = f'{OUTF}/student_{n}_rep{r}.pt'
        if not os.path.exists(p): print('missing', p); continue
        STU[(n, r)] = load_student(p); s = SC[f'{n}_rep{r}_tealeaf']; THR[(n, r)] = thr_at_sens(s, dte)
        a = roc_auc_score(dte, s); pub = RJ[f'{n}_rep{r}']['tealeaf_test']['auc']
        assert abs(a - pub) < 1e-6, f'saved scores for {n} rep{r} do not match the robustness JSON'
        print('%s rep%d loaded | our-test AUC %.4f (published %.4f) | 95%%-sens threshold %.4f' % (n, r, a, pub, THR[(n, r)]))
assert len(STU) == 9, 'expected nine students'

@torch.no_grad()
def scores(m, pils, bs=128):
    out = []
    for i in range(0, len(pils), bs):
        x = torch.stack([TO_T(p) for p in pils[i:i + bs]]).to(device)
        out.append(torch.softmax(m(x), 1)[:, 1].cpu().numpy())
    return np.concatenate(out)


In [ ]:
# CELL 7 — CHECK: re-score 96 of our test images with the loaded weights; must match the saved scores
SPL = json.load(open(SPLIT)); TP = SPL['test_paths']
local = lambda p: os.path.join(TEA_ROOT, *p.split('/')[-2:])
chk = np.random.default_rng(3).choice(len(TP), 96, replace=False)
pil = [otsu_crop(load_rgb(local(TP[i])))[0] for i in chk]
for key in (('S0', 0), ('S1', 1)):
    s = scores(STU[key], pil); ref = SC[f'{key[0]}_rep{key[1]}_tealeaf'][chk]
    d = np.abs(s - ref); print('%s rep%d: median |score diff| %.4f, 95th pct %.4f' % (*key, np.median(d), np.percentile(d, 95)))
    assert np.median(d) < 0.05, 'loaded weights or preprocessing do not reproduce the saved scores: stop'
    if np.median(d) >= 0.02: print('  note: small differences come from the crop recipe (external notebook), not the weights')
print('OK: weights and preprocessing reproduce the published scores.')


In [ ]:
# CELL 8 — blind transfer: score every student on both external sets at its fixed threshold
R = {}
for tag, S in SETS.items():
    for (n, r), m in STU.items():
        s = scores(m, S['crops']); f = s >= THR[(n, r)]
        R[f'{tag}/{n}/rep{r}'] = dict(auc=float(roc_auc_score(S['y'], s)), sens=float(f[S['y'] == 1].mean()),
                                      spec=float((~f)[S['y'] == 0].mean()),
                                      flag_rate_by_class={c: float(f[S['cls'] == c].mean()) for c in np.unique(S['cls'])})
SUM = {}
for tag in SETS:
    SUM[tag] = {'n': int(len(SETS[tag]['y'])), 'n_healthy': int((SETS[tag]['y'] == 0).sum())}
    for n in NAMES:
        rows = [R[f'{tag}/{n}/rep{r}'] for r in REPS]
        SUM[tag][n] = {k: [float(np.mean([x[k] for x in rows])), float(np.std([x[k] for x in rows], ddof=1))] for k in ('auc', 'sens', 'spec')}
    SUM[tag]['paired_S1_minus_S0_auc'] = [R[f'{tag}/S1/rep{r}']['auc'] - R[f'{tag}/S0/rep{r}']['auc'] for r in REPS]
    SUM[tag]['paired_S1_minus_S0_spec'] = [R[f'{tag}/S1/rep{r}']['spec'] - R[f'{tag}/S0/rep{r}']['spec'] for r in REPS]
    print('\n%s (n=%d, healthy %d) — threshold fixed at 95%% sensitivity on our test split' % (tag, SUM[tag]['n'], SUM[tag]['n_healthy']))
    for n in NAMES:
        v = SUM[tag][n]; print('  %s: AUC %.3f ± %.3f | sens %.3f ± %.3f | spec %.3f ± %.3f' % (n, *v['auc'], *v['sens'], *v['spec']))
    print('  S1 − S0 per seed: AUC', np.round(SUM[tag]['paired_S1_minus_S0_auc'], 3), '| spec', np.round(SUM[tag]['paired_S1_minus_S0_spec'], 3))
json.dump(dict(per_model=R, summary=SUM, _meta=dict(
    models='experiment_F students S0/S1/S2 x seeds 0-2 (inference only)',
    inputs='External_1 = East West University YOLO leaf boxes (+6% pad); External_2 = Allynugger whole frame; 224x224, JPEG q95',
    threshold='per student, 95% sensitivity on the TeaLeafDiseaseBD test split (saved scores)')),
    open(OUT, 'w'), indent=1)
print('\nsaved', OUT, '-> upload this file')


In [ ]:
# CELL 9 — for item 2: list the final External dataset 1 files so you can upload the right ones
for p in sorted(glob.glob(f'{EXT_OUT}/*')):
    info = ''
    if p.endswith('.json'):
        try:
            j = json.load(open(p)); r = j.get('EWU_box_probe', {})
            c = j.get('recalibration', {}).get('EWU_box', {}).get('curve', {})
            info = 'EWU n=%s | 10/class %s | 20/class %s' % (r.get('n'), (c.get('10') or {}).get('auc'), (c.get('20') or {}).get('auc'))
        except Exception as e: info = str(e)[:60]
    elif p.endswith('.npz'):
        try: info = 'rows=%d' % len(np.load(p)['y'])
        except Exception as e: info = str(e)[:60]
    print('%-55s %8.1f KB  %s  %s' % (os.path.basename(p), os.path.getsize(p) / 1024,
          time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), info))
print('\nUpload the external_results_v3.json whose EWU n is 3007 (10/class AUC 0.973, 20/class 0.983) and feat_EWU_box.npz with 3007 rows.')
